# ICS 604: APPLIED DATA SCIENCE

- ## Decision Trees
- ## Model Selection
  - ### Cross-validation

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Decision Trees for Classification

In a classification tree, the goal is to partition the data into groups that are as homogeneous as possible with respect to the target class. For example, if we want to group together “reddish” circles, we might consider splitting the data based on a feature such as radius. The key question becomes: which value of the radius should we choose as the splitting threshold? A good split is one that separates the data into subsets (child nodes) where each subset contains observations that are mostly from a single class.

<center><img src="https://github.com/blueberrymusic/Deep-Learning-A-Visual-Approach/blob/main/Figures/Images/11-22.png?raw=true" width="600"></center>
<center><small>Image Source: https://github.com/blueberrymusic/Deep-Learning-A-Visual-Approach/</small></center><br>

To determine the best split, we evaluate how “pure” or homogeneous the resulting subsets are. A perfectly pure node contains observations from only one class, while a highly mixed node contains a diverse set of classes. Decision trees use quantitative measures of impurity (or conversely, homogeneity) to compare different possible splits and select the one that yields the best separation.

One common measure of impurity is **entropy**, which originates from information theory. Entropy measures the amount of uncertainty or disorder in a dataset. If a node contains a mix of classes, its entropy is high; if it contains only one class, its entropy is zero. Formally, the entropy of a node is defined as:

$$E_{node} =- \sum_{i=1}^C p_i log_2(p_i)$$

where $p_i$ is the proportion of samples belonging to class $i$, and $C$ is the total number of classes.

When a node is split into two children (e.g., left and right based on a radius threshold), we compute the combined entropy of the split as a weighted sum of the entropies of the child nodes:

$$ E_{split} = n_{left} * E_{left} + n_{right} * E_{right} $$ 

where $n_{left}$ and $n_{right}$ represent the proportions (or counts) of samples in the left and right child nodes, respectively.

The effectiveness of a split is then measured using **information gain**, which quantifies how much the entropy decreases as a result of the split:

$$ Gain = E_{root} - E_{split}$$

A higher information gain indicates a better split, as it means the resulting subsets are more homogeneous than the original node. The decision tree algorithm evaluates many possible thresholds (such as different radius values) and selects the one that maximizes this gain.

While entropy and information gain are widely used, other impurity measures can also guide the splitting process. A popular alternative is the **Gini impurity**, which is computationally simpler and often used in practice. Regardless of the specific measure, the underlying principle remains the same: choose splits that produce the most homogeneous child nodes.

<center><img src="https://github.com/blueberrymusic/Deep-Learning-A-Visual-Approach/blob/main/Figures/Images/11-14.png?raw=true" width="400"></center>
<center><small>Image Source: https://github.com/blueberrymusic/Deep-Learning-A-Visual-Approach/</small></center>

<center><img src="https://github.com/blueberrymusic/Deep-Learning-A-Visual-Approach/blob/main/Figures/Images/11-15.png?raw=true" width="600"></center>
<center><small>Image Source: https://github.com/blueberrymusic/Deep-Learning-A-Visual-Approach/</small></center>

### Classification Tree Example: Iris Dataset

A classic example for understanding classification trees is the Iris dataset, which contains measurements of iris flowers from three different species. Each observation includes features such as sepal length, sepal width, petal length, and petal width. The task is to correctly classify each flower into one of the three species based on these measurements.

A decision tree approaches this problem by recursively splitting the data based on feature thresholds. For instance, one of the first and most informative splits often involves petal length. Flowers with very small petal lengths are almost always setosa, so the tree can immediately separate this class with a simple rule like: “if petal length $\leq$ 2.46, classify as setosa.” This creates a pure node, since all samples in that branch belong to the same class.

After isolating one class, the tree continues splitting the remaining data to distinguish between the other two species (*versicolor* and *virginica*). It might next consider petal width or another feature, choosing a threshold that best separates these two classes. At each step, the algorithm evaluates multiple candidate splits and selects the one that minimizes impurity, ensuring that each new node becomes more homogeneous.

<center><img src="https://scikit-learn.org/stable/_images/iris.svg" width="700">
<center><small>Source: https://scikit-learn.org/stable/modules/tree.html</small></center><br>

Visually, this process can be interpreted as partitioning the feature space into rectangular regions. Each split corresponds to a decision boundary (a vertical or horizontal line in a 2D feature plot), gradually carving the space into areas where one class dominates. The final tree is a sequence of simple, interpretable rules that collectively define these regions.

One of the strengths of classification trees is their interpretability. The resulting model can be read as a flowchart: starting from the root, each internal node poses a question about a feature (e.g., “Is petal length $\leq$< 2.46?”), and each leaf node provides a class prediction. This makes it easy to understand how the model arrives at its decisions, unlike many more complex machine learning models.

However, trees can also become overly complex if allowed to grow without constraint, leading to overfitting. In practice, techniques such as limiting the tree depth or pruning are used to ensure that the model generalizes well to new, unseen data.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree

iris = load_iris()
X = iris.data
y = iris.target

tree_clf = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_clf.fit(X, y)

plt.figure(figsize=(10, 8))
plot_tree(tree_clf, 
          feature_names=iris.feature_names,
          class_names=iris.target_names,
          rounded=True,
          filled=True)
plt.show()

In [ ]:
# Use graphviz for visualization

from graphviz import Source
from sklearn.tree import export_graphviz

dot_data = export_graphviz(
        tree_clf, out_file=None,
        feature_names=iris.feature_names,
        class_names=iris.target_names,
        rounded=True,
        filled=True
    )

Source(dot_data)

In [ ]:
print(f"Feature importances:\n  {tree_clf.feature_importances_}\n")

n_feat = X.shape[1]
plt.figure(figsize=(8, 4))
plt.bar(np.arange(n_feat), tree_clf.feature_importances_, align='center')
plt.xticks(np.arange(n_feat), iris.feature_names, rotation=45)
plt.xlabel("Features")
plt.ylabel("Feature importance")
plt.show()

## Decision Tree Regressor

A decision tree regressor extends the idea of classification trees to continuous target variables. Instead of predicting a class label, the model predicts a numerical value, making it well-suited for capturing complex, non-linear relationships in data. By recursively splitting the feature space, the tree can approximate highly irregular patterns without requiring explicit assumptions about the underlying function.

At each node, the algorithm searches for the best feature and threshold to split the data, just as in classification trees. However, the objective is different: rather than maximizing class purity, the regression tree aims to minimize prediction error. Typically, this is done by reducing the variance of the target values within each node. A good split produces child nodes where the target values are as similar as possible.

Once the tree is fully grown, each leaf node contains a subset of the training data. The prediction for any new input is simply the average (or sometimes median) of the target values of the samples in that leaf. This means that the model produces **piecewise constant predictions**, where the input space is divided into regions and each region is assigned a fixed value.

Visually, this results in a step-like function when plotted against a single feature. Instead of a smooth curve, the model approximates the relationship using flat segments, with abrupt changes at the split boundaries. Despite this simplicity, regression trees can model complex patterns effectively, especially when combined into ensemble methods such as random forests or gradient boosting.

One limitation of decision tree regressors is their tendency to overfit the training data, particularly when the tree becomes very deep. To address this, techniques such as limiting the maximum depth, requiring a minimum number of samples per leaf, or pruning the tree are commonly applied. These constraints help the model generalize better while still capturing the essential structure of the data.

<center><img src="https://www.dropbox.com/scl/fi/h1qpjhizry6g3ci8d6ugj/data_X.png?rlkey=5u68kzf2rezrx0iw9hxis9urd&st=ko9gf3px&dl=1" alt="drawing" width="300"/></center>

<center><img src="https://www.dropbox.com/scl/fi/g12xnfso3xj5qmweh9pbf/regression_tree.png?rlkey=cs1f0a1bfiaenplgp2obwb0o6&st=1rnn98t9&dl=1" alt="drawing" width="700"/></center>

In [ ]:
# Quadratic training set + noise
np.random.seed(10)
m = 150
X = np.random.rand(m, 1)
y = 5 * (X - 0.5) ** 2
y = y + np.random.randn(m, 1) / 5

plt.plot(X, y, 'b.')
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg.fit(X, y)

reg_data = export_graphviz(
        tree_reg, out_file=None,
        feature_names=["x1"],
        rounded=True,
        filled=True
    )
Source(reg_data)

In [ ]:
# Communities and Crime dataset
# https://archive.ics.uci.edu/dataset/183/communities+and+crime

url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data'
crime = pd.read_csv(url, header=None, na_values=['?'])
crime.head()

In [ ]:
crime.shape

```
@attribute state numeric
@attribute county numeric
@attribute community numeric
@attribute communityname string
@attribute fold numeric
@attribute population numeric
@attribute householdsize numeric
@attribute racepctblack numeric
@attribute racePctWhite numeric
@attribute racePctAsian numeric
@attribute racePctHisp numeric
@attribute agePct12t21 numeric
@attribute agePct12t29 numeric
@attribute agePct16t24 numeric
@attribute agePct65up numeric
@attribute numbUrban numeric
@attribute pctUrban numeric
@attribute medIncome numeric
@attribute pctWWage numeric
@attribute pctWFarmSelf numeric
@attribute pctWInvInc numeric
@attribute pctWSocSec numeric
@attribute pctWPubAsst numeric
@attribute pctWRetire numeric
@attribute medFamInc numeric
@attribute perCapInc numeric
@attribute whitePerCap numeric
@attribute blackPerCap numeric
@attribute indianPerCap numeric
@attribute AsianPerCap numeric
@attribute OtherPerCap numeric
@attribute HispPerCap numeric
@attribute NumUnderPov numeric
...
```

In [ ]:
# drop columns: state, county, community, communityname, fold
crime.drop([0, 1, 2, 3, 4], axis=1, inplace=True)

# remove rows with any missing values
crime.dropna(inplace=True)

# check the shape
crime.shape

In [ ]:
### Col 127 is {ViolentCrimesPerPop}: Per Capita Violent Crimes
### The value we'd like to predict
X = crime.drop(127, axis=1)
y = crime[127]
y.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

In [ ]:
print(f'Number of training samples: {len(y_train)}')
print(f'Number of testing samples: {len(y_test)}')

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor()
tree_reg.fit(X_train, y_train)

In [ ]:
y_train_pred = tree_reg.predict(X_train)

print(y_train_pred[0:10])
print(y_train.tolist()[0:10])

In [ ]:
from sklearn.metrics import mean_squared_error

print(np.sqrt(mean_squared_error(y_train_pred.reshape(-1, 1), y_train)))
print(np.round(np.sqrt(mean_squared_error(y_train_pred.reshape(-1, 1),  y_train)), 2))

In [ ]:
# tree_reg.tree_ gives access to the underlying tree.
# In scikit-learn, leaf nodes are marked by children_left == -1

n_leaves = sum(tree_reg.tree_.children_left == -1)
print(f'Number of leaf nodes: {n_leaves}')
print(f'Depth of the tree: {tree_reg.get_depth()}')

In [ ]:
print(y_train.count())

## Overfitting

Even if a decision tree regressor achieves an RMSE of 0 on the training data, this does not mean the model is truly accurate or “perfect.” In fact, this is usually a warning sign. A zero training error often indicates that the model has **overfit** the data — meaning it has essentially memorized the training examples rather than learned a general pattern.

This happens because decision trees are highly flexible. If left unconstrained, they can keep splitting until each leaf node contains very few (or even just one) data points. At that point, the model can reproduce the training targets exactly, leading to an RMSE of 0. However, this “perfect fit” is misleading: the model is not learning the underlying relationship, but rather storing the data.

The real issue becomes clear when we evaluate the model on unseen data. When computing the RMSE on the test set, we observe a higher error (e.g., 0.23 in this case). This gap between training performance and test performance is a hallmark of overfitting. The model performs exceptionally well on familiar data but struggles to generalize to new inputs.

<center><img src="https://www.dropbox.com/scl/fi/vh4nx9641yi8ywtn8qor4/overfitting.png?rlkey=4dcicnq1grjdbjvv2vxme1uoh&st=9becmrrh&dl=1" alt="drawing" width="600"/></center>

In [ ]:
y_test_pred = tree_reg.predict(X_test)

print(np.round(np.sqrt(mean_squared_error(y_test_pred.reshape(-1,1),  y_test)), 2))

### Assessing a Model's Generalization Power

In real-world machine learning, the primary objective is not to perfectly reproduce the training data, but to build models that perform well on **unseen data**. A model that simply memorizes past observations offers little practical value, since it cannot reliably make predictions in new situations. True success lies in capturing the underlying patterns in the data so that predictions remain accurate beyond the training set.

For this reason, model selection — choosing the best algorithm or tuning its parameters — must be guided by performance on data that was not used during training. However, the test set should be treated as a final benchmark only. If it is used repeatedly during model tuning, the model may begin to implicitly “fit” the test set, undermining its role as an unbiased estimate of generalization performance.

#### Decision Trees and Train-Test Split

Decision trees are particularly sensitive to this issue because they are highly flexible models with many tunable parameters, such as maximum depth, minimum samples per leaf, and splitting criteria. These parameters can significantly influence performance metrics like RMSE.

Another complication is that the test error can vary depending on how the data is split. Different train-test splits may produce noticeably different results, especially when the dataset is not very large. This variability makes it difficult to confidently assess which parameter settings truly perform best.

In [ ]:
tree_reg = DecisionTreeRegressor(max_depth=3)
tree_reg.fit(X_train, y_train)
y_test_pred = tree_reg.predict(X_test)
print(np.round(np.sqrt(mean_squared_error(y_test_pred.reshape(-1,1),  y_test)), 2))

In [ ]:
np.random.seed(42)

In [ ]:
rmse = []
for i in range(500):
    X_train, X_test, y_train, y_test = train_test_split(X, y)
    tree_reg = DecisionTreeRegressor(max_depth=3)
    tree_reg.fit(X_train, y_train)
    y_test_pred = tree_reg.predict(X_test)
    rmse.append(np.round(np.sqrt(mean_squared_error(y_test_pred.reshape(-1,1),  y_test)), 2))

plt.figure(figsize=(8, 4))
plt.hist(rmse, edgecolor='black', linewidth=1.2);

In [ ]:
np.random.seed(42)

In [ ]:
rmse = []
for i in range(500):
    X_train, X_test, y_train, y_test = train_test_split(X, y)
    tree_reg = DecisionTreeRegressor(max_depth=1)
    tree_reg.fit(X_train, y_train)
    y_test_pred = tree_reg.predict(X_test)
    rmse.append(np.round(np.sqrt(mean_squared_error(y_test_pred.reshape(-1,1),  y_test)), 2))

plt.figure(figsize=(8, 4))
plt.hist(rmse, edgecolor='black', linewidth=1.2);

### Train / Validation / Test Approach

To address this, the dataset is often divided into three parts:

- A **training set**, used to fit the model
- A **validation set**, used to evaluate and tune model parameters
- A **test set**, reserved strictly for final evaluation

In this approach, the model is trained on the training subset, and different parameter configurations are compared based on their performance on the validation set. Once the best configuration is selected, the model is evaluated on the test set to estimate its generalization error and to compare it with other models.

#### Limitations of the Validation Approach

While effective, this method has some drawbacks. Splitting the data reduces the number of observations available for training, which can negatively impact model performance — especially for complex models like decision trees that benefit from more data. Additionally, the validation error may not perfectly reflect the true test error, and dedicating a portion of the data solely for validation can feel inefficient.

### K-Fold Cross Validation

A more robust alternative is **K-fold cross-validation**, which makes better use of the available data. Instead of creating a single validation set, the training data is divided into K equal-sized subsets (or “folds”). The model is then trained and evaluated K times:

- In each iteration, K − 1 folds are used for training
- The remaining fold is used for validation
- This process repeats until each fold has served as the validation set once

The performance metrics (such as RMSE) from all K iterations are then averaged to produce a more stable estimate of model performance.

This approach reduces the variability associated with a single train-test split and allows every observation to be used for both training and validation at different stages. After identifying the best parameters through cross-validation, the final model is trained on the entire training dataset and evaluated once on the untouched test set.

In practice, a common choice is K=10, which provides a good balance between computational efficiency and reliable performance estimation.

<center><img src="https://www.dropbox.com/scl/fi/j79w1uwr8ewbwk0shtqgh/cross_validation.png?rlkey=1nz09miyu8ani6xte7lzk8i6b&st=kphlbd1u&dl=1" alt="drawing" style="width:800px;"/></center>

In [ ]:
from sklearn.model_selection import cross_val_score

tree_reg = DecisionTreeRegressor(max_depth=3)

scores = cross_val_score(tree_reg, X_train, y_train,
                        scoring='neg_mean_squared_error', cv=10)

tree_reg_rmse = np.sqrt(np.mean(-scores))

print(np.round(tree_reg_rmse, 2))

#### Scoring the K Data Chunks

When performing K-fold cross-validation, each of the K subsets (or folds) must be evaluated using a consistent scoring metric. In many regression tasks, we naturally think in terms of Mean Squared Error (MSE), where lower values indicate better performance. However, this creates a small mismatch with the design of tools like `cross_val_score`, which expect a similarity score — meaning that higher values should correspond to better models.

To resolve this, a scoring function called `"neg_mean_squared_error"` is used. This is simply the negative of the MSE:

$$ -1 \times MSE $$

By multiplying MSE by −1, we effectively flip the scale: smaller errors (which are good) become larger negative values in magnitude but closer to zero, and thus higher scores (less negative) correspond to better models. This allows the cross-validation framework to remain consistent across different types of metrics.

This design choice ensures that all scoring functions within the framework follow the same convention: higher is better, regardless of whether the underlying metric is naturally a loss (like MSE) or a similarity measure.

See link below for available scoring functions for regression: 
http://scikit-learn.org/stable/modules/model_evaluation.html

### Testing

Once all models have been trained and their parameters carefully tuned (using validation techniques such as cross-validation), the test set is finally used to evaluate their true generalization performance. This step provides an unbiased estimate of how each model will perform on completely unseen data, which is the ultimate goal in any machine learning task.

The evaluation process is straightforward. For each candidate model, we generate predictions on the test data and compare these predictions to the actual observed values using an appropriate metric, such as RMSE. The model with the smallest error is typically considered the best, as it demonstrates the strongest ability to generalize beyond the training data.

However, selecting a model is not purely a numerical decision — it also depends on the application. Different domains have different tolerance levels for error. For example, a model used in a patient-facing healthcare application may require extremely high accuracy and reliability, whereas a model predicting something like algae blooms might tolerate a higher degree of error. Context matters when determining whether a model’s performance is “good enough.”

An important aspect of this approach is that we prioritize generalization performance over simplicity. A more complex model is not inherently penalized if it delivers significantly better results on unseen data. That said, when two models achieve similar performance, it is generally preferable to choose the simpler one — fewer parameters often mean better interpretability, lower computational cost, and reduced risk of overfitting in future use.

In summary, the test phase is where all modeling decisions are validated. It ensures that the selected model is not just tailored to the training data, but is truly capable of performing well in real-world scenarios.